# Lab 2 — Decoding (EP01)

In Lab 1 we counted the tokens a model reads and writes. In this lab we look at how the model **chooses** the next token.

**What you'll learn**
1. Get Gemma's real next-token probabilities (`logprobs`)
2. Write greedy, temperature, top-k, top-p and min-p yourself, one function at a time
3. Sample tokens with our own code, then compare with the real API's `temperature`

Every probability and model answer comes from a real run — your numbers may differ slightly from the video.

> Reference: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §1.12 — CC BY-SA 4.0

## 0. Setup

You need Python 3.10+ (or Colab) and `OPENROUTER_API_KEY` — how to set up the key, and why we chose this model, are in **Lab 1**

In [ ]:
%pip install -q "openai>=1.50" python-dotenv

In [ ]:
# ---- from Lab 1: set up the client ----
import os, math, random
from collections import Counter
from openai import OpenAI

def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key())
MODEL = "google/gemma-4-26b-a4b-it"

In [ ]:
PROVIDER = {"order": ["google-vertex/global"], "allow_fallbacks": False}
EXTRA = {"reasoning": {"enabled": False}, "provider": PROVIDER}

## 1. Real next-token probabilities

Ask for `logprobs=True, top_logprobs=10` with `max_tokens=1` → the API returns the 10 most likely tokens for the answer's first position,
with a **logprob** (the log of the probability) — `math.exp(logprob)` turns it back into a probability from 0 to 1

In [ ]:
def next_token_probs(prompt):
    resp = client.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": prompt}],
        max_tokens=1, logprobs=True, top_logprobs=10, extra_body=EXTRA,
    )
    top = resp.choices[0].logprobs.content[0].top_logprobs
    probs = {t.token: math.exp(t.logprob) for t in top}
    return dict(sorted(probs.items(), key=lambda kv: -kv[1]))

In [ ]:
def show(probs, n=6):
    for tok, p in list(probs.items())[:n]:
        print(f"{tok!r:>14} {p:6.3f} {'█' * round(p * 40)}")

In [ ]:
PARIS = next_token_probs("Complete with one word only. The capital of France is")
show(PARIS)

In [ ]:
FOOD = next_token_probs("Complete with one word only. I enjoy eating")
show(FOOD)

## 2. Greedy and temperature

**Greedy** = always take the top token · **Temperature** = raise each probability to the power `1/T`, then make them sum to 1
(the same result as the textbook's formula) — `T < 1` makes the top tokens taller, `T > 1` brings everything closer together

In [ ]:
def greedy(probs):
    return max(probs, key=probs.get)

print(greedy(PARIS), greedy(FOOD))

In [ ]:
def normalize(weights):
    total = sum(weights.values())
    return {t: w / total for t, w in weights.items()}

def temperature(probs, T):
    return normalize({t: p ** (1 / T) for t, p in probs.items()})

In [ ]:
for T in [0.5, 1, 2]:
    print(f"T = {T}")
    show(temperature(FOOD, T), n=4)

## 3. Cutting the tail: top-k, top-p, min-p

| Method | Keeps |
|---|---|
| top-k | always the first `k` |
| top-p | sorted high to low, keep until the total reaches `p` |
| min-p | every token with at least `m ×` the top token's probability |

In [ ]:
def top_k(probs, k):
    return normalize(dict(list(probs.items())[:k]))

def top_p(probs, p):
    keep, total = {}, 0
    for t, x in probs.items():
        keep[t] = x
        total += x
        if total >= p:
            break
    return normalize(keep)

def min_p(probs, m):
    top = max(probs.values())
    return normalize({t: x for t, x in probs.items() if x >= m * top})

In [ ]:
for name, probs in [("Paris", PARIS), ("eating", FOOD)]:
    print(name)
    print("  top-k=5   :", list(top_k(probs, 5)))
    print("  top-p=0.95:", list(top_p(probs, 0.95)))
    print("  min-p=0.05:", list(min_p(probs, 0.05)))

## 4. Actually sampling

`random.choices` samples by weight · `random.seed(0)` makes our sampling give the same result every run

In [ ]:
def sample(probs):
    return random.choices(list(probs), weights=list(probs.values()))[0]

random.seed(0)
for T in [0.3, 1.5]:
    picks = [sample(top_p(temperature(FOOD, T), 0.95)) for _ in range(20)]
    print(f"T = {T}:", Counter(picks))

## 5. Comparing with the API's `temperature`

Ask Gemma the same question 10 times at `temperature=0` and `1.5`, letting the server sample
(tested 2026-11-09: the same `seed` on the server can still give different answers — an API seed makes results "close", not guaranteed "identical")

In [ ]:
for T in [0, 1.5]:
    answers = []
    for _ in range(10):
        r = client.chat.completions.create(
            model=MODEL, messages=[{"role": "user", "content": "Complete with one word only. I enjoy eating"}],
            max_tokens=3, temperature=T, extra_body=EXTRA,
        )
        answers.append(r.choices[0].message.content.strip())
    print(f"T = {T}:", Counter(answers))

## Summary

| What we saw | From this lab |
|---|---|
| The model gives every candidate a probability | Paris ≈ 1.0 · pizza ≈ 0.8, then food, everything |
| Greedy always takes the top token | Paris, pizza every time |
| Temperature changes the shape | T = 0.5 pizza nearly all · T = 2 about half |
| top-k is a fixed count · top-p / min-p adapt to confidence | Paris: 5 vs 1 · eating: 3 |
| API `temperature=0` ≈ greedy | pizza 10/10 |

**Exercise:** write a prompt of your own whose answer you expect to be spread out, then see how many tokens `top_p(…, 0.95)` keeps.
Then apply T = 2 before cutting and count again.

▶ **Next: Lab 3 — model names, providers, quantization and modality** — why the same Gemma from different providers gives different results